In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import joblib
import json
import os
from scipy import stats

In [3]:
# Load artifacts 
best_model = joblib.load("D:/credit-card-crosssell/models/xgb_propensity.pkl")
encoders   = joblib.load("D:/credit-card-crosssell/models/encoders.pkl")

with open("D:/credit-card-crosssell/models/selected_features.json") as f:
    selected = json.load(f)

with open("D:/credit-card-crosssell/models/best_params.json") as f:
    best_params = json.load(f)

with open("D:/credit-card-crosssell/models/model_results.json") as f:
    results = json.load(f)

print(f"Selected features : {len(selected)}")
print(f"AUC Test          : {results['auc_test']}")
print(f"AUC OOT           : {results['auc_oot']}")

Selected features : 27
AUC Test          : 0.9033
AUC OOT           : 0.9042


In [12]:
# Load current data 
oot = pd.read_csv("D:/credit-card-crosssell/data/feature_store_oot.csv")

# Load label OOT
oot_label = pd.read_csv("D:/credit-card-crosssell/data/label_oot.csv")
y_oot = oot_label['converted'].values

drop_cols  = ['customer_id', 'observation_date']
X_oot  = oot.drop(columns=drop_cols)

# Encode 
X_oot_encoded = X_oot.copy()
for col in ['gender', 'province']:
    X_oot_encoded[col] = encoders[col].transform(X_oot[col])

# Score 
scores_oot = best_model.predict_proba(X_oot_encoded[selected])[:, 1]

result_oot = pd.DataFrame({
    'customer_id'     : oot['customer_id'],
    'observation_date': oot['observation_date'],
    'label': oot_label['converted'],
    'propensity_score': scores_oot.round(4),
}).sort_values('propensity_score', ascending=False).reset_index(drop=True)

In [14]:
cost_per_contact = 83_000
revenue_per_conv = 15_000_000
thresholds       = np.arange(0.1, 0.95, 0.01)

best_roi       = -np.inf
best_threshold = 0.5
results_list   = []

for t in thresholds:
    y_pred = (scores_oot >= t).astype(int)
    
    tp = ((y_pred == 1) & (y_oot == 1)).sum()
    fp = ((y_pred == 1) & (y_oot == 0)).sum()
    fn = ((y_pred == 0) & (y_oot == 1)).sum()
    tn = ((y_pred == 0) & (y_oot == 0)).sum()
    
    roi       = tp * revenue_per_conv - (tp + fp) * cost_per_contact
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0

    results_list.append({
        'threshold': round(t, 2),
        'roi'      : roi,
        'precision': round(precision, 4),
        'recall'   : round(recall, 4),
        'tp'       : tp,
        'fp'       : fp,
        'contacted': tp + fp,
    })

    if roi > best_roi:
        best_roi       = roi
        best_threshold = t

results_df = pd.DataFrame(results_list)

print(f"Best threshold : {best_threshold:.2f}")
print(f"Best ROI       : {best_roi:,.0f} VND")
print()
print(results_df[results_df['threshold'] == round(best_threshold, 2)])

Best threshold : 0.24
Best ROI       : 30,949,197,000 VND

    threshold          roi  precision  recall    tp   fp  contacted
14       0.24  30949197000     0.7318     1.0  2079  762       2841


In [15]:
# export
result_oot.to_csv("D:/credit-card-crosssell/data/result_oot.csv")